In [1]:
import re
import json
import aiohttp
import asyncio
import numpy as np
import pandas as pd
from tqdm import tqdm

# Prepare Entity Linkers

In [2]:
from refined.inference.processor import Refined

REFINED_MODEL = Refined.from_pretrained(model_name='wikipedia_model_with_numbers', entity_set="wikidata")

def get_refined_entities(text, refined=REFINED_MODEL):
    return [{
        'id': span.__dict__['predicted_entity'].wikidata_entity_id,
        'label': span.__dict__['predicted_entity'].wikipedia_entity_title,
        'text': span.__dict__['text'],
        'span': tuple([span.__dict__['start'], span.__dict__['ln']]),
        'score': span.__dict__['entity_linking_model_confidence_score']
        } for span in refined.process_text(text) if span.predicted_entity is not None]

/Users/amalekseev/Documents/Thesis KGQA/kgqa_venv/lib/python3.11/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/amalekseev/Documents/Thesis KGQA/entity/ReFinED/src/refined/model_components/refined_model.py:626: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted b

## Entitiy Linking

In [3]:
def read_dataset(filepath):
    return json.load(open(filepath, "r", encoding="utf-8"))["dataset"]

In [4]:
dataset_paths = {
    dataset_name: f'data/datasets/{dataset_name}/{dataset_name}_test.json' for dataset_name in ['rubq', 'qald', 'pat', 'lcquad_2.0']
}

for dataset_name, filepath in dataset_paths.items():
    dataset = read_dataset(filepath)
    for item in tqdm(dataset):
        item['refined'] = get_refined_entities(item['en_question'])

    with open(f'data/datasets/{dataset_name}/{dataset_name}_test_we.json', 'w', encoding='utf-8') as f:
        json.dump(dataset, f, ensure_ascii=False, indent=4)

  0%|                                                   | 0/480 [00:00<?, ?it/s]/Users/amalekseev/Documents/Thesis KGQA/entity/ReFinED/src/refined/inference/processor.py:293: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/Users/amalekseev/Documents/Thesis KGQA/kgqa_venv/lib/python3.11/site-packages/torch/amp/autocast_mode.py:266: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn(
100%|███████████████████████████████████████| 4541/4541 [02:46<00:00, 27.35it/s]
